# LTSM Tuning

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import numpy as np
import math
import random
from dataclasses import dataclass

import numpy as np
import polars as pl
import matplotlib.pyplot as plt

# Для нормализации данных и расчета метрик
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
from pathlib import Path

# PyTorch для построения и обучения нейросетей
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader

def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else ("mps" if torch.mps.is_available() else "cpu"))
print("device:", device)

device: mps


In [5]:
data_dir = Path("../data/")
dataset = "g5_2xlarge_6h_after_2024-07_with_features.parquet"

df = pl.read_parquet(data_dir / dataset)

In [6]:


# параметры
SEQ_LEN = 28
HORIZON = 4
BATCH_SIZE = 32
EPOCHS = 50

# нормализация
scaler = StandardScaler()
feature_cols_lstm = [c for c in df.columns if c not in ["datetime"]]
data = df[feature_cols_lstm].to_numpy().astype(np.float32)

# Dataset
class TimeSeriesDataset(Dataset):
    def __init__(self, data, seq_len, horizon, target_idx):
        self.data = data
        self.seq_len = seq_len
        self.horizon = horizon
        self.target_idx = target_idx

    def __len__(self):
        return len(self.data) - self.seq_len - self.horizon + 1

    def __getitem__(self, idx):
        x = self.data[idx : idx + self.seq_len]
        y = self.data[idx + self.seq_len + self.horizon - 1, self.target_idx]
        return torch.tensor(x), torch.tensor(y)

# LSTM модель
class LSTMModel(nn.Module):
    def __init__(self, input_size, hidden_size=64, num_layers=2, dropout=0.2):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            dropout=dropout,
            batch_first=True
        )
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(out[:, -1, :]).squeeze()

In [7]:
target_idx = feature_cols_lstm.index("cost")
split_idx = int(len(data) * 0.8)

train_data = scaler.fit_transform(data[:split_idx])
test_data = scaler.transform(data[split_idx:])

train_dataset = TimeSeriesDataset(train_data, SEQ_LEN, HORIZON, target_idx)
test_dataset = TimeSeriesDataset(test_data, SEQ_LEN, HORIZON, target_idx)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

In [ ]:
import optuna

def objective(trial):
    hidden_size = trial.suggest_categorical("hidden_size", [64, 128, 256])
    num_layers = trial.suggest_int("num_layers", 1, 3)
    dropout = trial.suggest_float("dropout", 0.1, 0.4)
    lr = trial.suggest_float("lr", 1e-4, 1e-2, log=True)
    seq_len = trial.suggest_categorical("seq_len", [14, 28, 56])

    # пересоздаём датасет с новым seq_len
    train_dataset = TimeSeriesDataset(train_data, seq_len, HORIZON, target_idx)
    test_dataset = TimeSeriesDataset(test_data, seq_len, HORIZON, target_idx)
    train_loader = DataLoader(train_dataset, batch_size=32, shuffle=False)
    test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

    model = LSTMModel(
        input_size=data.shape[1],
        hidden_size=hidden_size,
        num_layers=num_layers,
        dropout=dropout
    ).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.MSELoss()

    for epoch in range(30):  # меньше эпох для скорости
        model.train()
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            loss = criterion(model(X_batch), y_batch)
            loss.backward()
            optimizer.step()

    model.eval()
    y_true, y_pred = [], []
    with torch.no_grad():
        for X_batch, y_batch in test_loader:
            pred = model(X_batch.to(device)).cpu().numpy().reshape(-1)
            y_pred.extend(pred)
            y_true.extend(y_batch.numpy().reshape(-1))
    cost_std = scaler.scale_[target_idx]
    cost_mean = scaler.mean_[target_idx]
    y_true = np.array(y_true) * cost_std + cost_mean
    y_pred = np.array(y_pred) * cost_std + cost_mean

    return mean_absolute_error(y_true, y_pred)

study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=50)

print("Best params:", study.best_params)
print("Best MAE:", study.best_value)

[I 2026-05-30 08:26:03,621] A new study created in memory with name: no-name-e401010e-3123-4109-af7a-effc687637e3
/Users/kryuchkov_ds/vscode/.venv/lib/python3.12/site-packages/torch/nn/modules/rnn.py:1013: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.2523804584288916 and num_layers=1
  super().__init__("LSTM", *args, **kwargs)
[I 2026-05-30 08:26:10,034] Trial 0 finished with value: 0.018694482698998247 and parameters: {'hidden_size': 64, 'num_layers': 1, 'dropout': 0.2523804584288916, 'lr': 0.0013011054331851582, 'seq_len': 56}. Best is trial 0 with value: 0.018694482698998247.
[I 2026-05-30 08:26:22,282] Trial 1 finished with value: 0.05820567542412537 and parameters: {'hidden_size': 256, 'num_layers': 2, 'dropout': 0.3767478226272566, 'lr': 0.00015230646607467614, 'seq_len': 56}. Best is trial 0 with value: 0.018694482698998247.
[I 2026-05-30 08:26:27,371] Trial 2 finished with v

Best params: {'hidden_size': 64, 'num_layers': 1, 'dropout': 0.3062587613127175, 'lr': 0.00087510170922976, 'seq_len': 14}
Best MAE: 0.010059994757296997


# Сохраним лучшие параметры

In [11]:
import yaml

best_params = study.best_params

config = {
    "model": {
        "hidden_size": best_params["hidden_size"],
        "num_layers": best_params["num_layers"],
        "dropout": best_params["dropout"],
        "input_size": data.shape[1],
    },
    "training": {
        "learning_rate": best_params["lr"],
        "epochs": 50,
        "batch_size": BATCH_SIZE,
        "seq_len": best_params["seq_len"],
        "horizon": HORIZON,
    },
    "metrics": {
        "best_mae": round(study.best_value, 4),
    }
}

with open("../configs/training.yaml", "w") as f:
    yaml.dump(config, f, default_flow_style=False, allow_unicode=True)

print("saved configs/training.yaml")

saved configs/training.yaml
